# Philippine Commodity Price Tracker: Real Data Exploration

Welcome to **Phase 1: Real Data Engineering & Exploration**!

> **100% Real Official Data**: All records in this database (`data/commodities.duckdb`) are extracted directly from official **Department of Agriculture price monitoring** *Daily Price Index* PDF bulletins published on [da.gov.ph](https://www.da.gov.ph/price-monitoring/). Zero synthetic data.

### Objectives of this Notebook
1. **Inspect Data Provenance & Audit Trail**: Query the `commodity_prices` and `fetch_log` tables in `DuckDB`.
2. **Analyze Official DA Price Trends**: Inspect real prevailing retail prices for staples (Rice, Chicken, Beef, Pork, Fish, Vegetables, Sugar, Cooking Oil) in Metro Manila (NCR) wet markets.
3. **Experiment with Time-Series Analytics**: Test SQL aggregations, rolling moving averages, and Plotly visualizations to guide our upcoming **Phase 2 (Analytics)** and **Phase 3 (Forecasting)** modules.

---

## 1. Connect to DuckDB & Verify Real Data Tables

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.renderers.default = "notebook_connected"

# Connect to DuckDB
db_path = Path("../data/commodities.duckdb") if Path("../data/commodities.duckdb").exists() else Path("data/commodities.duckdb")
con = duckdb.connect(str(db_path))
print(f"Connected to DuckDB: {db_path.resolve()}")

# Inspect tables in the database
con.execute("SHOW TABLES").df()

Connected to DuckDB: /home/leinnarf/Project/Philippine Commodity Price Tracker/data/commodities.duckdb


,name
0,commodity_prices
1,fetch_log


## 2. Ingestion Audit Trail (`fetch_log`)
Let's inspect the recent automated downloads of Department of Agriculture price monitoring PDF bulletins, SHA-256 hashes, and record counts.

In [2]:
log_query = """
SELECT 
    target_date,
    status,
    http_status,
    records_ingested,
    detail,
    content_sha256,
    fetched_at
FROM fetch_log
ORDER BY target_date DESC;
"""
con.execute(log_query).df()

,target_date,status,http_status,records_ingested,detail,content_sha256,fetched_at
0,2026-10-21,not_published,404,0,Bulletin not found or not yet published on DA ...,None,2026-10-21 19:51:24.400619
1,2026-10-20,ok,200,157,Parsed from cached PDF (157 records),98c3a0497884eb65ff8aff9dd2128aca49d6ed8dc10832...,2026-10-21 19:50:58.075044
2,2026-10-19,ok,200,77,Successfully downloaded and parsed (77 records),751644099bd7e0fdaaf4d2b4b61a924339978d0ec04d8f...,2026-10-21 19:50:55.503220
3,2026-10-17,ok,200,89,Successfully downloaded and parsed (89 records),1dc4782ff490c36a85cb60ad74d98c06c8cc798ed69278...,2026-10-21 19:50:27.287770
4,2026-10-16,ok,200,95,Successfully downloaded and parsed (95 records),fe3732fe41c7604d0e611fc131c0c4497de46ad1c36c40...,2026-10-21 19:49:59.459090
5,2026-10-15,ok,200,91,Successfully downloaded and parsed (91 records),4226f2a6afec76aa5cd8ab2299d04ff6c8ef3f6a3858f4...,2026-10-21 19:49:26.296383
6,2026-10-14,ok,200,94,Successfully downloaded and parsed (94 records),d07f7c762f283fa222efed1509813aff5640d485580320...,2026-10-21 19:48:58.351092
7,2026-10-13,ok,200,95,Successfully downloaded and parsed (95 records),d95b9a64a460c96955757a15c9ca1f8fea2c4bb4f9bf9e...,2026-10-21 19:48:30.322395
8,2026-10-12,ok,200,94,Successfully downloaded and parsed (94 records),5afcc74e20a4d7937d4afd512800ffe9918de0681e47a3...,2026-10-21 19:48:21.202684
9,2026-10-10,ok,200,89,Successfully downloaded and parsed (89 records),ec4edce8008da12952c3e52b5191996249b77f34efda5f...,2026-10-21 19:47:53.266557


## 3. Real Dataset Summary & Provenance Verification
Confirm that `is_synthetic` is `FALSE` across the entire database, and check total observation dates.

In [3]:
summary_query = """
SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT commodity) AS distinct_commodities,
    COUNT(DISTINCT commodity_group) AS distinct_categories,
    MIN(date)::VARCHAR AS earliest_date,
    MAX(date)::VARCHAR AS latest_date,
    COUNT(DISTINCT date) AS total_market_days,
    SUM(CASE WHEN is_synthetic THEN 1 ELSE 0 END) AS synthetic_rows,
    SUM(CASE WHEN NOT is_synthetic THEN 1 ELSE 0 END) AS real_da_rows
FROM commodity_prices;
"""
con.execute(summary_query).df()

,total_records,distinct_commodities,distinct_categories,earliest_date,latest_date,total_market_days,synthetic_rows,real_da_rows
0,1148,148,9,2026-10-07,2026-10-20,12,0.0,1148.0


## 4. Category Breakdown & Average Market Prices
Let's see the food groups tracked by the DA and average prevailing prices in Metro Manila wet markets.

In [4]:
cat_query = """
SELECT 
    commodity_group AS category,
    COUNT(DISTINCT commodity) AS distinct_items,
    COUNT(*) AS total_observations,
    ROUND(AVG(price_prevailing), 2) AS category_avg_php,
    MIN(price_prevailing) AS lowest_price_php,
    MAX(price_prevailing) AS highest_price_php
FROM commodity_prices
GROUP BY commodity_group
ORDER BY category_avg_php DESC;
"""
con.execute(cat_query).df()

,category,distinct_items,total_observations,category_avg_php,lowest_price_php,highest_price_php
0,Meat & Poultry,57,336,389.61,8.81,756.85
1,Fish,15,28,340.12,154.03,613.64
2,Other,37,369,242.46,8.22,638.70
3,Spices,9,19,232.14,124.29,380.00
4,Highland Vegetables,19,194,212.24,60.49,377.88
5,Fruits,10,20,147.01,66.40,418.33
6,Lowland Vegetables,7,77,118.74,60.36,210.72
7,Other Basic Goods,10,92,99.09,22.14,165.00
8,Rice & Cereals,13,13,78.57,45.33,194.29


## 5. Most Frequently Monitored Commodities
Inspect commodities with the highest daily observation coverage.

In [5]:
top_items_query = """
SELECT 
    commodity,
    commodity_group,
    unit,
    COUNT(DISTINCT date) AS days_tracked,
    ROUND(AVG(price_prevailing), 2) AS avg_price_php,
    MIN(price_prevailing) AS min_price_php,
    MAX(price_prevailing) AS max_price_php
FROM commodity_prices
GROUP BY commodity, commodity_group, unit
ORDER BY days_tracked DESC, avg_price_php DESC
LIMIT 20;
"""
con.execute(top_items_query).df()

,commodity,commodity_group,unit,days_tracked,avg_price_php,min_price_php,max_price_php
0,"Beef Forequarter, Local",Meat & Poultry,kg,12,484.16,451.43,500.00
1,"Beef Brisket, Local",Meat & Poultry,kg,12,446.52,442.56,449.77
2,"Beef Fore Limb, Local",Meat & Poultry,kg,12,439.01,383.33,470.00
3,"Beef Chuck, Local",Meat & Poultry,kg,12,431.85,429.75,434.25
4,"Beef Brisket, Imported",Meat & Poultry,kg,12,389.88,386.00,395.00
5,Cooking Oil (Coconut),Other Basic Goods,bottle (1L),12,153.76,152.06,155.62
6,Cooking Oil (Spring),Other Basic Goods,bottle (1L),12,140.41,120.00,143.33
7,Cooking Oil (Palm),Other Basic Goods,bottle (1L),12,101.94,100.10,103.06
8,Sugar (Brown),Other Basic Goods,kg,12,72.28,71.70,73.19
9,Cooking Oil (Coconut),Other Basic Goods,bottle (350ml),12,59.05,58.29,59.86


## 6. Real Time-Series Analysis: Whole Dressed Chicken
Let's analyze daily retail prices for **Whole Chicken, Local** across Metro Manila markets.

In [6]:
target_commodity = "Whole Chicken, Local"

ts_query = f"""
SELECT 
    date,
    commodity,
    variety,
    unit,
    price_prevailing,
    source_url
FROM commodity_prices
WHERE commodity = '{target_commodity}'
ORDER BY date ASC;
"""
ts_df = con.execute(ts_query).df()
ts_df["date"] = pd.to_datetime(ts_df["date"])

# Rolling 5-day moving average (matching ~1 government work week)
ts_df["sma_5d"] = ts_df["price_prevailing"].rolling(window=5, min_periods=1).mean()

# Day-over-day change
ts_df["dod_change_php"] = ts_df["price_prevailing"].diff()
ts_df["dod_change_pct"] = ts_df["price_prevailing"].pct_change() * 100

ts_df

,date,commodity,variety,unit,price_prevailing,source_url,sma_5d,dod_change_php,dod_change_pct
0,2026-10-07,"Whole Chicken, Local",Fully Dressed,kg,200.14,https://www.da.gov.ph/wp-content/uploads/2026/...,200.140,NaN,NaN
1,2026-10-07,"Whole Chicken, Local",Magnolia,kg,209.49,https://www.da.gov.ph/wp-content/uploads/2026/...,204.815,9.35,4.671730
2,2026-10-07,"Whole Chicken, Local",Bounty Fresh,kg,192.50,https://www.da.gov.ph/wp-content/uploads/2026/...,200.710,-16.99,-8.110172
3,2026-10-08,"Whole Chicken, Local",Fully Dressed,kg,200.59,https://www.da.gov.ph/wp-content/uploads/2026/...,200.680,8.09,4.202597
4,2026-10-08,"Whole Chicken, Local",Magnolia,kg,209.34,https://www.da.gov.ph/wp-content/uploads/2026/...,202.412,8.75,4.362132
5,2026-10-08,"Whole Chicken, Local",Bounty Fresh,kg,192.50,https://www.da.gov.ph/wp-content/uploads/2026/...,200.884,-16.84,-8.044330
6,2026-10-09,"Whole Chicken, Local",Fully Dressed,kg,200.07,https://www.da.gov.ph/wp-content/uploads/2026/...,199.000,7.57,3.932468
7,2026-10-09,"Whole Chicken, Local",Magnolia,kg,208.86,https://www.da.gov.ph/wp-content/uploads/2026/...,202.272,8.79,4.393462
8,2026-10-09,"Whole Chicken, Local",Bounty Fresh,kg,194.00,https://www.da.gov.ph/wp-content/uploads/2026/...,200.954,-14.86,-7.114814
9,2026-10-10,"Whole Chicken, Local",Magnolia,kg,208.96,https://www.da.gov.ph/wp-content/uploads/2026/...,200.878,14.96,7.711340


## 7. Plotly Trend Chart: Real Philippine Retail Prices
Interactive time-series graph with range slider and moving average.

In [9]:
fig = go.Figure()

# Real prevailing price line
fig.add_trace(go.Scatter(
    x=ts_df["date"],
    y=ts_df["price_prevailing"],
    mode="lines+markers",
    name="Department of Agriculture Price Monitoring Prevailing Price",
    line=dict(color="#1e3a8a", width=2.5),
    marker=dict(size=6, color="#1e3a8a")
))

# 5-day Moving Average (1-week trend)
fig.add_trace(go.Scatter(
    x=ts_df["date"],
    y=ts_df["sma_5d"],
    mode="lines",
    name="5-Day Moving Average",
    line=dict(color="#d97706", width=2, dash="dash")
))

# Formal layout styling
fig.update_layout(
    title=dict(
        text=f"Official Department of Agriculture Price Monitoring Retail Price Trend: {target_commodity} (NCR Wet Markets)",
        font=dict(size=16, family="sans-serif")
    ),
    xaxis=dict(
        title="Market Date",
        showgrid=True,
        gridcolor="#e2e8f0",
        rangeslider=dict(visible=True)
    ),
    yaxis=dict(
        title="Price (PHP per kg)",
        showgrid=True,
        gridcolor="#e2e8f0",
        tickprefix="₱"
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.show()

## 8. Guiding Questions for Phase 2 (Analytics) & Phase 3 (Forecasting)

Now that we have real Department of Agriculture price monitoring data:
1. **Commodity Selection**: Which specific staples do you want featured on the main dashboard cards? (e.g., `Whole Chicken, Local`, `Beef Brisket, Local`, `Cooking Oil (Palm)`, `Sugar (Brown)`, `Ampalaya`).
2. **Moving Averages**: Does a 5-day (1 work week) and 10-day (2 work weeks) SMA work best for daily government market reports?
3. **Next Steps**: When you are ready, we will implement **Phase 2 (Analytics)** to compute rolling price spreads, top movers, and category indices directly from these real numbers.

In [8]:
# Scratchpad: Write any custom queries against real Department of Agriculture price monitoring data
# Example: con.execute("SELECT * FROM commodity_prices WHERE commodity_group = 'Meat & Poultry' LIMIT 10").df()
